# Assignment 5.1: Vehicle Routing Problem with Time Windows (VRPTW)

## 🎯 Learning Objectives

In this assignment, you will:
- Understand real-world vehicle routing problems
- Learn about time window constraints
- Implement domain-specific GA operators for routing
- Compare GA with classical heuristics (Nearest Neighbor, Clarke-Wright)
- Visualize delivery routes on geographic maps
- Apply GAs to solve industrial-scale logistics problems

This is a **real-world case study** similar to problems solved by Amazon, UPS, and food delivery companies!

---

## 📦 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Import VRPTW modules
from delivery_problem import Customer, Depot, Vehicle, Route, DeliveryProblem
from delivery_ga import delivery_genetic_algorithm
from baselines import nearest_neighbor_vrptw, clarke_wright_vrptw, sweep_vrptw
from generate_datasets import generate_clustered_customers, generate_random_customers

# Import visualization
sys.path.append('../ga_toolkit')
from visualization import plot_convergence, plot_convergence_comparison

np.random.seed(42)

print("✅ Libraries imported successfully!")
print("\nThis notebook demonstrates GA for real-world logistics optimization.")

---

## 🎓 2. Understanding Vehicle Routing with Time Windows

### 2.1 - What is VRPTW?

The **Vehicle Routing Problem with Time Windows (VRPTW)** is a classic logistics optimization problem:

**Given:**
- A depot (warehouse/distribution center)
- N customers with delivery demands
- Each customer has a time window [earliest, latest] for delivery
- M vehicles with limited capacity

**Find:**
- Routes for each vehicle
- Minimize total distance traveled
- Satisfy all constraints:
  - Each customer visited exactly once
  - Vehicle capacity not exceeded
  - Time windows respected
  - Routes start and end at depot

### 2.2 - Real-World Applications

- **Package Delivery**: Amazon Prime, UPS, FedEx
- **Food Delivery**: Uber Eats, DoorDash (hot food has strict time windows!)
- **Service Routing**: Plumbers, electricians, healthcare
- **Waste Collection**: Garbage trucks with schedules

### 2.3 - Why is it Hard?

- **NP-Hard complexity**: No polynomial-time exact solution
- **Multiple objectives**: Distance, time, vehicle count
- **Complex constraints**: Capacity, time windows, route feasibility
- **Large scale**: 50-200 customers typical in real scenarios

**GAs are excellent for VRPTW** because they can:
- Handle complex constraints naturally
- Find good solutions in reasonable time
- Be customized with domain knowledge

---

## 🏗️ 3. Create a VRPTW Problem Instance

Let's create a realistic delivery scenario with clustered customers (like urban neighborhoods).

In [ ]:
# Generate a problem instance
print("Generating VRPTW problem instance...")
print("=" * 70)

# Create depot (distribution center)
depot = Depot(
    location=(40.7589, -73.9851),  # Times Square, NYC (example)
    name="Main Depot"
)

# Generate 30 clustered customers (realistic urban scenario)
customers = generate_clustered_customers(
    n_customers=30,
    n_clusters=3,
    depot_location=depot.location,
    radius_km=10.0,
    time_window_width=120,  # 2-hour windows
    seed=42
)

print(f"\n✅ Created {len(customers)} customers in 3 clusters")
print(f"   Depot location: {depot.location}")
print(f"   Service area radius: ~10 km")

# Show sample customers
print("\nSample customers:")
for i in range(min(3, len(customers))):
    c = customers[i]
    print(f"  Customer {c.customer_id}:")
    print(f"    Location: {c.location}")
    print(f"    Demand: {c.demand} units")
    print(f"    Time Window: {c.time_window[0]:.0f} - {c.time_window[1]:.0f} min")
    print(f"    Service Time: {c.service_time:.0f} min")

### 3.1 - Define Vehicle Fleet

In [ ]:
# Create vehicle fleet
n_vehicles = 5
vehicles = [
    Vehicle(
        vehicle_id=i,
        capacity=100,  # 100 units capacity
        max_distance=150.0  # 150 km max range
    )
    for i in range(n_vehicles)
]

print(f"\nFleet: {n_vehicles} vehicles")
print(f"  Capacity: {vehicles[0].capacity} units each")
print(f"  Max Range: {vehicles[0].max_distance} km")

# Calculate total demand
total_demand = sum(c.demand for c in customers)
total_capacity = sum(v.capacity for v in vehicles)
print(f"\nDemand Analysis:")
print(f"  Total demand: {total_demand} units")
print(f"  Total capacity: {total_capacity} units")
print(f"  Utilization: {100 * total_demand / total_capacity:.1f}%")

### 3.2 - Create the Delivery Problem

In [ ]:
# Create the complete problem
problem = DeliveryProblem(
    depot=depot,
    customers=customers,
    vehicles=vehicles
)

print("\n" + "=" * 70)
print("VRPTW Problem Created!")
print("=" * 70)
print(f"  Depot: {problem.depot.name}")
print(f"  Customers: {len(problem.customers)}")
print(f"  Vehicles: {len(problem.vehicles)}")
print(f"  Distance Matrix: {problem.distance_matrix.shape}")
print(f"  Time Matrix: {problem.time_matrix.shape}")

---

## 📊 4. Visualize the Problem

Let's visualize the customer locations and time windows.

In [ ]:
# Plot customer locations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Geographic distribution
lats = [c.location[0] for c in customers]
lons = [c.location[1] for c in customers]
demands = [c.demand for c in customers]

scatter = ax1.scatter(lons, lats, c=demands, s=100, cmap='YlOrRd', 
                     alpha=0.7, edgecolors='black', linewidth=1.5)
ax1.scatter(depot.location[1], depot.location[0], s=400, c='blue', 
           marker='s', label='Depot', edgecolors='black', linewidth=2, zorder=5)

ax1.set_xlabel('Longitude', fontsize=12)
ax1.set_ylabel('Latitude', fontsize=12)
ax1.set_title('Customer Geographic Distribution', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax1, label='Demand (units)')

# Right: Time windows
time_starts = [c.time_window[0] for c in customers]
time_ends = [c.time_window[1] for c in customers]
customer_ids = [c.customer_id for c in customers]

for i, cid in enumerate(customer_ids[:15]):  # Show first 15 for clarity
    ax2.barh(i, time_ends[i] - time_starts[i], left=time_starts[i],
            height=0.6, alpha=0.7, edgecolor='black', linewidth=1)
    ax2.text(time_starts[i] - 10, i, f'C{cid}', fontsize=9, va='center')

ax2.set_xlabel('Time (minutes from depot opening)', fontsize=12)
ax2.set_ylabel('Customer Index', fontsize=12)
ax2.set_title('Time Windows (First 15 Customers)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n📊 Left: Customers clustered in neighborhoods (color = demand)")
print("📊 Right: Time window constraints for each customer")

---

## 🔧 5. Exercise: Implement Route Evaluation

Before running the GA, let's understand how routes are evaluated.

### Exercise 1: Calculate Total Route Distance

Given a route (sequence of customer indices), calculate the total distance traveled.

**Instructions:**
- Route starts and ends at depot (index 0)
- Use `problem.distance_matrix` to get distances
- Sum: depot → customer1 → customer2 → ... → depot

In [ ]:
def calculate_route_distance(route_customers, distance_matrix):
    """
    Calculate total distance of a route.
    
    Arguments:
    route_customers -- list of customer indices (not including depot)
    distance_matrix -- (n+1) x (n+1) matrix where index 0 is depot
    
    Returns:
    total_distance -- total distance in km
    """
    
    ### START CODE HERE ### (≈ 5-7 lines)
    
    # Start at depot (index 0)
    total_distance = 0.0
    current = 0  # depot
    
    # Visit each customer
    for customer_idx in route_customers:
        total_distance += distance_matrix[current, customer_idx]
        current = customer_idx
    
    # Return to depot
    total_distance += distance_matrix[current, 0]
    
    ### END CODE HERE ###
    
    return total_distance

In [ ]:
# Test your implementation
print("Testing route distance calculation:")
print("=" * 70)

# Simple test route: depot → customer 1 → customer 2 → depot
test_route = [1, 2]
test_distance = calculate_route_distance(test_route, problem.distance_matrix)

print(f"\nTest Route: Depot → Customer 1 → Customer 2 → Depot")
print(f"  Distance: {test_distance:.2f} km")

# Verify manually
manual_dist = (problem.distance_matrix[0, 1] + 
               problem.distance_matrix[1, 2] + 
               problem.distance_matrix[2, 0])
print(f"  Expected: {manual_dist:.2f} km")
print(f"  Match: {np.isclose(test_distance, manual_dist)}")

assert np.isclose(test_distance, manual_dist), "Distance calculation incorrect!"
print("\n✅ Route distance calculation correct!")

### Exercise 2: Check Time Window Feasibility

**Task:** Check if a route respects all customer time windows.

**Time window constraint:**
- Arrival time must be within [earliest, latest]
- If arrive early, wait until earliest time
- If arrive late, time window violated

**Instructions:**
1. Calculate arrival time at each customer
2. Check if within time window
3. Add service time before moving to next

In [ ]:
def check_time_windows(route_customers, problem):
    """
    Check if route satisfies time window constraints.
    
    Arguments:
    route_customers -- list of customer indices
    problem -- DeliveryProblem instance
    
    Returns:
    feasible -- True if all time windows satisfied
    total_violation -- total time window violation (minutes)
    """
    
    ### START CODE HERE ### (≈ 12-15 lines)
    
    current_time = 0  # Start at time 0 at depot
    current_location = 0  # Depot index
    total_violation = 0
    
    for customer_idx in route_customers:
        # Travel to customer
        travel_time = problem.time_matrix[current_location, customer_idx]
        current_time += travel_time
        
        # Get customer time window
        customer = problem.customers[customer_idx - 1]
        earliest, latest = customer.time_window
        
        # Check time window
        if current_time < earliest:
            # Arrive early, wait
            current_time = earliest
        elif current_time > latest:
            # Arrive late, violation!
            total_violation += (current_time - latest)
        
        # Service time
        current_time += customer.service_time
        current_location = customer_idx
    
    feasible = (total_violation == 0)
    
    ### END CODE HERE ###
    
    return feasible, total_violation

In [ ]:
# Test your implementation
print("Testing time window checking:")
print("=" * 70)

# Test with a short route
test_route = [1, 2, 3]
feasible, violation = check_time_windows(test_route, problem)

print(f"\nTest route: {test_route}")
print(f"  Feasible: {feasible}")
print(f"  Violation: {violation:.1f} minutes")

if feasible:
    print("\n✅ Route respects all time windows!")
else:
    print(f"\n⚠️ Route has {violation:.1f} min of time window violations")

print("\n✅ Time window check implemented!")

---

### Exercise 3: Check Capacity Feasibility

**Task:** Check if a route respects vehicle capacity.

**Capacity constraint:**
- Sum of all customer demands ≤ vehicle capacity

**Instructions:**
1. Sum demands of all customers in route
2. Compare with vehicle capacity
3. Calculate excess if over capacity

In [ ]:
def check_capacity(route_customers, problem, vehicle_capacity):
    """
    Check if route satisfies capacity constraint.
    
    Arguments:
    route_customers -- list of customer indices
    problem -- DeliveryProblem instance
    vehicle_capacity -- maximum capacity of vehicle
    
    Returns:
    feasible -- True if capacity not exceeded
    total_load -- total load on vehicle
    excess -- amount over capacity (0 if feasible)
    """
    
    ### START CODE HERE ### (≈ 5-7 lines)
    
    # Sum all demands
    total_load = 0
    for customer_idx in route_customers:
        customer = problem.customers[customer_idx - 1]
        total_load += customer.demand
    
    # Check capacity
    excess = max(0, total_load - vehicle_capacity)
    feasible = (excess == 0)
    
    ### END CODE HERE ###
    
    return feasible, total_load, excess

In [ ]:
# Test your implementation
print("Testing capacity checking:")
print("=" * 70)

# Test with a route
test_route = [1, 2, 3, 4, 5]
vehicle_cap = problem.vehicles[0].capacity

feasible, total_load, excess = check_capacity(test_route, problem, vehicle_cap)

print(f"\nTest route: {test_route}")
print(f"  Vehicle capacity: {vehicle_cap} units")
print(f"  Total load: {total_load} units")
print(f"  Utilization: {100 * total_load / vehicle_cap:.1f}%")
print(f"  Feasible: {feasible}")

if not feasible:
    print(f"  Excess load: {excess} units")

# Verify calculation
manual_load = sum(problem.customers[i-1].demand for i in test_route)
assert total_load == manual_load, "Load calculation incorrect!"

print("\n✅ Capacity check implemented!")

---

### Exercise 4: Evaluate Route Quality

**Task:** Calculate overall route quality (fitness) considering distance and constraint violations.

**Formula:**
$$\text{Quality} = -(\text{Distance} + \text{penalty\_weight} \times \text{Violations})$$

**Instructions:**
1. Calculate route distance
2. Check time window violations
3. Check capacity violations
4. Combine with penalties (higher quality = better)

In [ ]:
def evaluate_route_quality(route_customers, problem, vehicle_capacity, penalty_weight=100.0):
    """
    Evaluate overall route quality.
    
    Arguments:
    route_customers -- list of customer indices
    problem -- DeliveryProblem instance
    vehicle_capacity -- vehicle capacity
    penalty_weight -- penalty multiplier for violations
    
    Returns:
    quality -- route quality (higher is better, negative of cost)
    """
    
    ### START CODE HERE ### (≈ 8-10 lines)
    
    # Calculate distance
    distance = calculate_route_distance(route_customers, problem.distance_matrix)
    
    # Check time windows
    tw_feasible, tw_violation = check_time_windows(route_customers, problem)
    
    # Check capacity
    cap_feasible, total_load, cap_excess = check_capacity(route_customers, problem, vehicle_capacity)
    
    # Calculate total cost (distance + penalties)
    total_cost = distance + penalty_weight * (tw_violation + cap_excess)
    
    # Quality is negative cost (GA maximizes)
    quality = -total_cost
    
    ### END CODE HERE ###
    
    return quality

In [ ]:
# Test your implementation
print("Testing route quality evaluation:")
print("=" * 70)

# Test with a route
test_route = [1, 2, 3]
vehicle_cap = problem.vehicles[0].capacity

quality = evaluate_route_quality(test_route, problem, vehicle_cap, penalty_weight=100.0)

print(f"\nTest route: {test_route}")
print(f"  Quality score: {quality:.2f}")
print(f"  (Higher is better, negative means cost)")

# Compare two routes
route_a = [1, 2]
route_b = [1, 2, 3, 4, 5]

quality_a = evaluate_route_quality(route_a, problem, vehicle_cap)
quality_b = evaluate_route_quality(route_b, problem, vehicle_cap)

print(f"\nComparing routes:")
print(f"  Route A {route_a}: quality = {quality_a:.2f}")
print(f"  Route B {route_b}: quality = {quality_b:.2f}")
print(f"  Better route: {'A' if quality_a > quality_b else 'B'}")

print("\n✅ Route quality evaluation implemented!")

---

## 🚀 6. Run Genetic Algorithm for VRPTW

Now let's use the specialized GA to solve this routing problem!

The GA uses domain-specific operators:
- **Route-aware crossover**: Preserves route structure
- **Swap mutation**: Exchanges customers between routes
- **Penalty-based fitness**: Soft constraints for time windows
- **Repair operators**: Fix infeasible solutions

In [ ]:
print("Running Genetic Algorithm for VRPTW...")
print("=" * 70)

# Run the specialized delivery GA
best_solution, best_fitness, history = delivery_genetic_algorithm(
    problem=problem,
    pop_size=100,
    max_generations=200,
    crossover_rate=0.8,
    mutation_rate=0.2,
    penalty_weight=1000.0,  # Heavy penalty for violations
    verbose=True
)

print("\n" + "=" * 70)
print("GA Optimization Complete!")
print("=" * 70)

### 6.1 - Analyze the Best Solution

In [ ]:
# Decode the solution
from delivery_ga import decode_solution

routes = decode_solution(best_solution, problem)

print("\n📊 BEST SOLUTION ANALYSIS")
print("=" * 70)
print(f"\nTotal Distance: {-best_fitness:.2f} km")  # Negative because GA maximizes
print(f"Vehicles Used: {len([r for r in routes if r.customers])}")

print("\nRoute Details:")
for i, route in enumerate(routes):
    if route.customers:  # Only show non-empty routes
        n_customers = len(route.customers)
        total_demand = sum(problem.customers[c-1].demand for c in route.customers)
        route_dist = calculate_route_distance(route.customers, problem.distance_matrix)
        
        print(f"\n  Vehicle {i + 1}:")
        print(f"    Customers: {route.customers[:10]}{'...' if n_customers > 10 else ''}")
        print(f"    Count: {n_customers} customers")
        print(f"    Load: {total_demand}/{problem.vehicles[i].capacity} units ({100*total_demand/problem.vehicles[i].capacity:.1f}%)")
        print(f"    Distance: {route_dist:.2f} km")

# Check constraint satisfaction
all_customers = set()
for route in routes:
    all_customers.update(route.customers)

expected_customers = set(range(1, len(problem.customers) + 1))
print(f"\n✅ All customers served: {all_customers == expected_customers}")
print(f"   Customers served: {len(all_customers)}/{len(problem.customers)}")

---

## 📈 7. Visualize Convergence

In [ ]:
# Plot GA convergence
plot_convergence(
    history,
    title="GA Convergence for VRPTW",
    figsize=(12, 6)
)

plt.show()

print("\n📊 The GA converged to a good solution!")
print(f"   Initial best fitness: {history['best_fitness'][0]:.2f}")
print(f"   Final best fitness: {history['best_fitness'][-1]:.2f}")
print(f"   Improvement: {100 * (history['best_fitness'][-1] - history['best_fitness'][0]) / abs(history['best_fitness'][0]):.1f}%")

---

## 🎯 8. Compare with Classical Heuristics

Let's compare the GA with traditional routing algorithms:
- **Nearest Neighbor (NN)**: Greedy, always visit closest customer
- **Clarke-Wright Savings**: Merge routes to maximize savings
- **Sweep Algorithm**: Angular sweep from depot

In [ ]:
print("Comparing with Classical Heuristics...")
print("=" * 70)

# Run baseline algorithms
print("\nRunning Nearest Neighbor...", end=" ")
nn_routes, nn_distance = nearest_neighbor_vrptw(problem)
print(f"Done! Distance: {nn_distance:.2f} km")

print("Running Clarke-Wright...", end=" ")
cw_routes, cw_distance = clarke_wright_vrptw(problem)
print(f"Done! Distance: {cw_distance:.2f} km")

print("Running Sweep Algorithm...", end=" ")
sweep_routes, sweep_distance = sweep_vrptw(problem)
print(f"Done! Distance: {sweep_distance:.2f} km")

ga_distance = -best_fitness

print("\n" + "=" * 70)
print("COMPARISON RESULTS")
print("=" * 70)
print(f"\n{'Algorithm':<20} {'Distance (km)':<15} {'Gap to GA':<15} {'Vehicles'}")
print("-" * 70)
print(f"{'Genetic Algorithm':<20} {ga_distance:<15.2f} {'-':<15} {len([r for r in routes if r.customers])}")
print(f"{'Nearest Neighbor':<20} {nn_distance:<15.2f} {f'+{100*(nn_distance-ga_distance)/ga_distance:.1f}%':<15} {len([r for r in nn_routes if r.customers])}")
print(f"{'Clarke-Wright':<20} {cw_distance:<15.2f} {f'+{100*(cw_distance-ga_distance)/ga_distance:.1f}%':<15} {len([r for r in cw_routes if r.customers])}")
print(f"{'Sweep Algorithm':<20} {sweep_distance:<15.2f} {f'+{100*(sweep_distance-ga_distance)/ga_distance:.1f}%':<15} {len([r for r in sweep_routes if r.customers])}")

print("\n💡 Positive gap means GA found a better (shorter) solution!")

### 8.1 - Visualize Comparison

In [ ]:
# Bar chart comparison
algorithms = ['GA', 'Nearest\nNeighbor', 'Clarke-\nWright', 'Sweep']
distances = [ga_distance, nn_distance, cw_distance, sweep_distance]
colors = ['green', 'orange', 'orange', 'orange']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(algorithms, distances, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

# Add value labels
for i, (bar, dist) in enumerate(zip(bars, distances)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{dist:.1f} km',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Total Distance (km)', fontsize=12, fontweight='bold')
ax.set_title('Algorithm Comparison - VRPTW', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(distances) * 1.15)

plt.tight_layout()
plt.show()

print(f"\n✅ GA achieves {100*(nn_distance-ga_distance)/nn_distance:.1f}% improvement over Nearest Neighbor!")
print(f"✅ GA achieves {100*(cw_distance-ga_distance)/cw_distance:.1f}% improvement over Clarke-Wright!")

---

## 🗺️ 9. Visualize Routes on Map

Let's visualize the best GA solution as delivery routes.

In [ ]:
# Plot routes
fig, ax = plt.subplots(figsize=(14, 10))

# Define colors for routes
route_colors = ['red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink']

# Plot each route
for i, route in enumerate(routes):
    if not route.customers:
        continue
    
    color = route_colors[i % len(route_colors)]
    
    # Build route coordinates: depot → customers → depot
    route_lats = [depot.location[0]]
    route_lons = [depot.location[1]]
    
    for customer_idx in route.customers:
        customer = problem.customers[customer_idx - 1]
        route_lats.append(customer.location[0])
        route_lons.append(customer.location[1])
    
    # Return to depot
    route_lats.append(depot.location[0])
    route_lons.append(depot.location[1])
    
    # Plot route line
    ax.plot(route_lons, route_lats, '-', color=color, linewidth=2, 
            alpha=0.6, label=f'Vehicle {i+1} ({len(route.customers)} customers)')
    
    # Plot customer points
    for j, customer_idx in enumerate(route.customers):
        customer = problem.customers[customer_idx - 1]
        ax.scatter(customer.location[1], customer.location[0], 
                  s=150, c=color, edgecolors='black', linewidth=1, zorder=5, alpha=0.8)
        ax.text(customer.location[1], customer.location[0], f'{j+1}', 
               ha='center', va='center', fontsize=8, fontweight='bold', color='white')

# Plot depot
ax.scatter(depot.location[1], depot.location[0], s=500, c='gold', 
          marker='*', edgecolors='black', linewidth=2, zorder=10, label='Depot')

ax.set_xlabel('Longitude', fontsize=12, fontweight='bold')
ax.set_ylabel('Latitude', fontsize=12, fontweight='bold')
ax.set_title(f'Best GA Solution - Total Distance: {ga_distance:.2f} km', 
            fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🗺️ Routes visualized! Each color represents one vehicle's route.")
print("   Numbers show visit sequence for each vehicle.")

---

## 💡 10. Key Insights and Real-World Impact

### What We Learned:

1. **GAs excel at complex routing problems** - Better than classical heuristics in most cases

2. **Domain knowledge matters** - Specialized operators (route crossover, repair) crucial

3. **Constraints can be soft** - Penalty functions handle time windows gracefully

4. **Visualization is critical** - Routes must make geographic sense

### Real-World Impact:

**Savings for a delivery company:**
- 30 customers/day, 250 working days/year
- GA saves ~10 km/day vs Nearest Neighbor
- Annual savings: 2,500 km × $0.50/km = **$1,250/vehicle/year**
- For 100-vehicle fleet: **$125,000/year** in fuel costs!

**Additional benefits:**
- Reduced CO2 emissions
- Better customer satisfaction (time windows)
- Fewer vehicles needed (capacity optimization)
- Driver workload balancing

### Why GAs Work Well for VRPTW:

✅ **Handle multiple objectives**: Distance, time, vehicle count  
✅ **Incorporate domain knowledge**: Route-aware operators  
✅ **Robust to problem variations**: Easy to add new constraints  
✅ **Find good solutions quickly**: Acceptable runtime for daily planning  
✅ **Population diversity**: Multiple good solutions to choose from  

---

## 🎯 11. Challenge Exercise - Larger Problem

**Challenge:** Solve a larger VRPTW instance and compare results!

**Instructions:**
1. Generate 50 customers instead of 30
2. Use 7 vehicles instead of 5
3. Run GA for 300 generations
4. Compare with baselines
5. Visualize the routes

In [ ]:
### YOUR CODE HERE ###

# Hint: Follow the structure from above:
# 1. Create depot
# 2. Generate 50 customers with generate_clustered_customers
# 3. Create 7 vehicles
# 4. Create problem
# 5. Run GA
# 6. Compare with baselines
# 7. Visualize

print("Challenge: Solve 50-customer VRPTW instance!")
print("=" * 70)

# Create larger problem
depot_large = Depot(location=(40.7589, -73.9851), name="Main Depot")
customers_large = generate_clustered_customers(
    n_customers=50,
    n_clusters=4,
    depot_location=depot_large.location,
    radius_km=15.0,
    seed=123
)
vehicles_large = [Vehicle(i, capacity=120, max_distance=200.0) for i in range(7)]
problem_large = DeliveryProblem(depot_large, customers_large, vehicles_large)

print(f"\nLarge problem: {len(customers_large)} customers, {len(vehicles_large)} vehicles")

# Run GA
print("\nRunning GA (this may take 1-2 minutes)...")
best_sol_large, best_fit_large, hist_large = delivery_genetic_algorithm(
    problem=problem_large,
    pop_size=150,
    max_generations=300,
    crossover_rate=0.8,
    mutation_rate=0.2,
    verbose=False
)

ga_dist_large = -best_fit_large
print(f"\n✅ GA Distance: {ga_dist_large:.2f} km")

# Compare with baselines
print("\nComparing with baselines...")
nn_routes_l, nn_dist_l = nearest_neighbor_vrptw(problem_large)
cw_routes_l, cw_dist_l = clarke_wright_vrptw(problem_large)

print(f"  Nearest Neighbor: {nn_dist_l:.2f} km (+{100*(nn_dist_l-ga_dist_large)/ga_dist_large:.1f}%)")
print(f"  Clarke-Wright: {cw_dist_l:.2f} km (+{100*(cw_dist_l-ga_dist_large)/ga_dist_large:.1f}%)")

print(f"\n🎉 GA savings on 50-customer problem: {nn_dist_l - ga_dist_large:.2f} km vs NN!")
print(f"   That's {100*(nn_dist_l-ga_dist_large)/nn_dist_l:.1f}% improvement!")

---

## 📚 12. Summary

### What You Accomplished:

✅ Understood real-world vehicle routing problems  
✅ Implemented route evaluation functions  
✅ Applied specialized GA to VRPTW  
✅ Compared GA with classical heuristics  
✅ Visualized delivery routes on geographic maps  
✅ Calculated real-world cost savings  
✅ Solved a production-grade logistics problem  

### Real-World Applications:

This same approach is used by:
- **Amazon Prime** - Package delivery optimization
- **UPS/FedEx** - Multi-depot routing with time windows
- **Uber Eats/DoorDash** - Dynamic food delivery (real-time VRPTW)
- **Waste Management** - Garbage truck routing
- **Field Service** - Technician scheduling and routing

### Next Steps:

1. **Add more constraints**: Driver breaks, multiple depots, pickup-delivery pairs
2. **Dynamic routing**: Handle new orders arriving in real-time
3. **Multi-objective**: Optimize distance AND time AND customer satisfaction
4. **Machine Learning integration**: Predict traffic, service times
5. **Production deployment**: API, database integration, scalability

---

## 🎉 Congratulations!

You've completed an **industrial-grade case study** in genetic algorithms!

You now have the skills to:
- Tackle complex real-world optimization problems
- Design domain-specific GA operators
- Compare ML methods with classical algorithms
- Quantify business impact of optimization

**You're ready to apply GAs in industry!** 🚀

---

### 📖 Additional Resources:

- **Code**: All source code in `Tutorial_5_Industrial_Case_Study/`
- **Tests**: Run `python public_tests.py` to validate
- **README**: Detailed documentation in tutorial README
- **Datasets**: `generate_datasets.py` for more problem instances

**Keep optimizing!** 🌟